In [1]:
import pandas as pd
import sqlalchemy

engine = sqlalchemy.create_engine("sqlite+pysqlite:///data/orders_and_users.db")
inspector = sqlalchemy.inspect(engine)
print(inspector.get_table_names())
inspector.get_columns("orders")

['customers', 'orders']


[{'name': 'InvoiceNo',
  'type': VARCHAR(),
  'nullable': True,
  'default': None,
  'primary_key': 1},
 {'name': 'CustomerID',
  'type': VARCHAR(),
  'nullable': True,
  'default': None,
  'primary_key': 0},
 {'name': 'Description',
  'type': VARCHAR(),
  'nullable': True,
  'default': None,
  'primary_key': 0},
 {'name': 'Quantity',
  'type': INTEGER(),
  'nullable': True,
  'default': None,
  'primary_key': 0},
 {'name': 'UnitPrice',
  'type': FLOAT(),
  'nullable': True,
  'default': None,
  'primary_key': 0},
 {'name': 'Category',
  'type': VARCHAR(),
  'nullable': True,
  'default': None,
  'primary_key': 0},
 {'name': 'Discount',
  'type': FLOAT(),
  'nullable': True,
  'default': None,
  'primary_key': 0},
 {'name': 'PaymentMethod',
  'type': VARCHAR(),
  'nullable': True,
  'default': None,
  'primary_key': 0}]

In [6]:
query = sqlalchemy.text("""
SELECT
    CustomerID,
    InvoiceNo,
    Quantity * UnitPrice * (1 - Discount) AS total_sum,
    SUM(Quantity * UnitPrice * (1 - Discount)) OVER (PARTITION BY CustomerID) AS customer_total,
    CASE
        WHEN SUM(Quantity * UnitPrice * (1 - Discount)) OVER (PARTITION BY CustomerID) < 500 THEN 'Bronze'
        WHEN SUM(Quantity * UnitPrice * (1 - Discount)) OVER (PARTITION BY CustomerID) < 1000 THEN 'Silver'
        WHEN SUM(Quantity * UnitPrice * (1 - Discount)) OVER (PARTITION BY CustomerID) < 1500 THEN 'Gold'
        ELSE 'Platinum'
    END AS spending_group
FROM orders
""")
task1 = pd.read_sql(query, con=engine)
display(task1)
task1.groupby('spending_group').agg(
    orders_count=('InvoiceNo', 'count'),
    customers_count=('CustomerID', 'nunique')
)

,CustomerID,InvoiceNo,total_sum,customer_total,spending_group
0,002c2626-e984-459b-b96e-f66b9bf1c8d4,INV100640,478.0944,832.3230,Silver
1,002c2626-e984-459b-b96e-f66b9bf1c8d4,INV100997,51.7440,832.3230,Silver
2,002c2626-e984-459b-b96e-f66b9bf1c8d4,INV100633,185.6421,832.3230,Silver
3,002c2626-e984-459b-b96e-f66b9bf1c8d4,INV100044,116.8425,832.3230,Silver
4,0145193c-cf85-436e-92cc-17cb5b0d04eb,INV100280,55.9736,893.3536,Silver
...,...,...,...,...,...
95,10bf93a6-29de-4f41-afd0-e62610ac2068,INV100843,66.3344,1694.8796,Platinum
96,10bf93a6-29de-4f41-afd0-e62610ac2068,INV100616,263.0880,1694.8796,Platinum
97,10bf93a6-29de-4f41-afd0-e62610ac2068,INV100881,125.2350,1694.8796,Platinum
98,10bf93a6-29de-4f41-afd0-e62610ac2068,INV100916,192.6744,1694.8796,Platinum


,orders_count,customers_count
spending_group,,
Bronze,14,5
Gold,23,5
Platinum,22,3
Silver,41,12


In [7]:
query = sqlalchemy.text("""
WITH order_totals AS (
    SELECT CustomerID, InvoiceNo, Quantity * UnitPrice * (1 - Discount) AS total_sum
    FROM orders
),
customer_totals AS (
    SELECT CustomerID, SUM(total_sum) AS customer_total
    FROM order_totals
    GROUP BY CustomerID
),
top_customer AS (
    SELECT CustomerID FROM customer_totals ORDER BY customer_total DESC LIMIT 1
)
SELECT
    ot.CustomerID,
    ot.InvoiceNo,
    ot.total_sum,
    SUM(ot.total_sum) OVER (ORDER BY ot.total_sum ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_sum
FROM order_totals ot
JOIN top_customer tc ON ot.CustomerID = tc.CustomerID
ORDER BY ot.total_sum
""")
task2 = pd.read_sql(query, con=engine)
task2

,CustomerID,InvoiceNo,total_sum,cumulative_sum
0,01eac1fb-42ac-4ad4-afd6-fe01d3e2dcf5,INV100298,35.2320,35.2320
1,01eac1fb-42ac-4ad4-afd6-fe01d3e2dcf5,INV100542,35.3628,70.5948
2,01eac1fb-42ac-4ad4-afd6-fe01d3e2dcf5,INV100665,59.2410,129.8358
3,01eac1fb-42ac-4ad4-afd6-fe01d3e2dcf5,INV100546,93.3000,223.1358
4,01eac1fb-42ac-4ad4-afd6-fe01d3e2dcf5,INV100740,126.2700,349.4058
5,01eac1fb-42ac-4ad4-afd6-fe01d3e2dcf5,INV100234,130.7232,480.1290
6,01eac1fb-42ac-4ad4-afd6-fe01d3e2dcf5,INV100511,178.9320,659.0610
7,01eac1fb-42ac-4ad4-afd6-fe01d3e2dcf5,INV100660,222.3760,881.4370
8,01eac1fb-42ac-4ad4-afd6-fe01d3e2dcf5,INV100290,409.2816,1290.7186
9,01eac1fb-42ac-4ad4-afd6-fe01d3e2dcf5,INV100283,583.2960,1874.0146


In [8]:
query = sqlalchemy.text("""
WITH product_totals AS (
    SELECT Category, Description, SUM(Quantity * UnitPrice * (1 - Discount)) AS product_total
    FROM orders
    GROUP BY Category, Description
)
SELECT
    Category,
    Description,
    product_total,
    SUM(product_total) OVER (
        PARTITION BY Category
        ORDER BY product_total
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative_sum
FROM product_totals
ORDER BY Category, product_total
""")
task3 = pd.read_sql(query, con=engine)
task3

,Category,Description,product_total,cumulative_sum
0,Accessories,USB-C Cable,1767.7218,1767.7218
1,Accessories,HDMI Adapter,2167.3234,3935.0452
2,Accessories,Power Bank,4401.7294,8336.7746
3,Electronics,Bluetooth Speaker,1189.5100,1189.5100
4,Electronics,LED Monitor,3679.3956,4868.9056
5,Electronics,Wireless Mouse,3854.9347,8723.8403
6,Gaming,Gaming Keyboard,1703.2076,1703.2076
7,Office,Laptop Stand,3099.0924,3099.0924


In [9]:
query = sqlalchemy.text("""
WITH customer_totals AS (
    SELECT CustomerID, SUM(Quantity * UnitPrice * (1 - Discount)) AS customer_total
    FROM orders
    GROUP BY CustomerID
)
SELECT
    CustomerID,
    customer_total,
    RANK() OVER (ORDER BY customer_total DESC) AS rank
FROM customer_totals
ORDER BY rank
""")
task4 = pd.read_sql(query, con=engine)
task4

,CustomerID,customer_total,rank
0,01eac1fb-42ac-4ad4-afd6-fe01d3e2dcf5,1874.0146,1
1,10bf93a6-29de-4f41-afd0-e62610ac2068,1694.8796,2
2,07a005e9-45b1-446b-a374-c8490f7c5dfa,1619.3216,3
3,037f2615-2d98-4d9d-ba88-1aa37263dc0e,1474.2984,4
4,04e44f80-9f4e-4fc9-a45c-a81097d07a02,1470.1668,5
5,0f4490c6-673c-4a29-a1fa-59dfebeb7714,1408.9779,6
6,09bffaf1-15ef-465d-9204-22e5f16c6f6e,1170.8984,7
7,07065bc8-5d9c-4177-82f9-f5e9b64444f1,1012.1663,8
8,0a0ad13a-2fac-484f-b54c-9607340f0524,963.6841,9
9,0818064c-782b-4b0e-9ceb-1517653713f9,938.4184,10
